# 교통 점수 기준 수립을 위한 탐색적 데이터 분석 (EDA)

이 노트는 매물과 교통 시설(지하철, 버스) 간의 접근성을 분석하여, '교통 온도' 점수 산정의 최적 기준과 가중치를 결정하기 위해 작성되었습니다.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 로드

In [ ]:
PROJECT_ROOT = os.path.abspath("../../../")
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "GraphDB_data")

def load_properties(limit=5000):
    geo_file = os.path.join(DATA_DIR, "land", "00_통합_원투룸.json")
    if not os.path.exists(geo_file): return pd.DataFrame()
    with open(geo_file, 'r', encoding='utf-8') as f: geo_data = json.load(f)
    df = pd.DataFrame(geo_data)
    df['lat'] = df['좌표_정보'].apply(lambda x: x.get('위도') if x else None)
    df['lon'] = df['좌표_정보'].apply(lambda x: x.get('경도') if x else None)
    return df.dropna(subset=['lat', 'lon']).sample(n=min(limit, len(df)), random_state=42)

def load_transport():
    transport = {}
    # Subway
    s_path = os.path.join(DATA_DIR, "subway_station", "지하철_노선도.csv")
    try: df = pd.read_csv(s_path, encoding='utf-8', on_bad_lines='skip')
    except: df = pd.read_csv(s_path, encoding='cp949', on_bad_lines='skip')
    transport['subway'] = df.dropna(subset=['역위도', '역경도']).rename(columns={'역위도': 'lat', '역경도': 'lon'})
    
    # Bus
    b_path = os.path.join(DATA_DIR, "bus_station", "bus_data_fixed.csv")
    try: df_b = pd.read_csv(b_path, encoding='utf-8')
    except: df_b = pd.read_csv(b_path, encoding='cp949')
    transport['bus'] = df_b[df_b['도시명'].str.contains("서울특별시", na=False)].dropna(subset=['위도', '경도']).rename(columns={'위도': 'lat', '경도': 'lon'})
    return transport

props = load_properties()
trans = load_transport()
print(f"Props: {len(props)}, Subway: {len(trans['subway'])}, Bus: {len(trans['bus'])}")

## 2. 분석 실행 (거리 및 개수)

In [ ]:
def to_xy(df):
    return np.column_stack([df['lon'].values * 88000, df['lat'].values * 111000])

# Subway Distance
tree_sub = cKDTree(to_xy(trans['subway']))
dists_sub, _ = tree_sub.query(to_xy(props), k=1)

# Bus Count (300m)
tree_bus = cKDTree(to_xy(trans['bus']))
counts_bus = [len(l) for l in tree_bus.query_ball_point(to_xy(props), r=300)]

# Visualization
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(dists_sub, bins=50, kde=True)
plt.title("지하철 거리 분포 (m)")

plt.subplot(1, 2, 2)
sns.histplot(counts_bus, bins=20, kde=True)
plt.title("300m 내 버스 정류장 수")
plt.show()